# Playground — `initialize_weights.py`'s F_sel/J_sel construction functions

Small interactive notebook to actually **run** and **look at** the three ways
`initialize_weights.py` builds `F_sel`/`J_sel` to go alongside AAV9's real,
MLP-recovered `F_viab`/`J_viab`:

- `initialize_correlated_weights`     — F_sel/J_sel share F_viab/J_viab's trend
- `initialize_anticorrelated_weights` — F_sel/J_sel share the OPPOSITE trend
- `initialize_indep_weights`          — F_sel/J_sel: same values, reshuffled (no trend)

Goals:
1. Check the `correlation`/`anticorrelation` knobs actually deliver the target Pearson
   correlation (they're a statistical construction over a small number of cells --
   140 for F, 8,400 for J -- so there's real sampling noise to see).
2. Look at what the three regimes actually LOOK like (heatmaps + scatter).
3. See what different `key`s do for the SAME correlation target.
4. The payoff: plug all three into a real `ProtocolV2` simulation and watch how the
   viability/selectivity relationship changes what survives directed evolution.

## 0. Setup

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Must run before the first `import jax` anywhere (including transitively via
# sequence_classesV1/analysisV1/initialize_weights) -- same convention as DE_loopV1.ipynb.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sequence_classesV1 import *
from analysisV1 import *
from initialize_weights import (
# GT swapped 2026-08-27: now loads the joint Potts-regression GT (AAV9_potts_regression.ipynb) instead of the naive group-means one -- cell outputs below were cleared since they were computed under the old GT; re-run this notebook before trusting any number in it.
    load_F_viab_aav9_potts, load_J_viab_aav9_potts,
    initialize_correlated_weights, initialize_anticorrelated_weights, initialize_indep_weights,
    NUM_AMINO_ACIDS, NUM_POSITIONS,
)

print(f"JAX backend: {jax.default_backend()} -- devices: {jax.devices()}")

## 1. Load `F_viab` / `J_viab` (real AAV9, recovered by ProfileMLP)

In [ ]:
# GT swapped 2026-08-27: now loads the joint Potts-regression GT (AAV9_potts_regression.ipynb) instead of the naive group-means one -- cell outputs below were cleared since they were computed under the old GT; re-run this notebook before trusting any number in it.
F_viab = load_F_viab_aav9_potts()
J_viab = load_J_viab_aav9_potts()

off_diag = ~np.eye(NUM_POSITIONS, dtype=bool)
print(f"F_viab shape: {F_viab.shape}")
print(f"J_viab shape: {J_viab.shape}  (off-diagonal entries: {int(off_diag.sum()) * NUM_AMINO_ACIDS**2:,})")

_ = plot_teacher_weights(F_viab, J_viab, title="F_viab / J_viab -- real AAV9 (ProfileMLP recovery)")
plt.show()

## 2. `initialize_correlated_weights` — does `correlation` land where it should?

For each target correlation, draw `n_keys` independent `F_sel`/`J_sel` and measure the
ACHIEVED Pearson r against `F_viab` (140 cells) and `J_viab`'s off-diagonal (8,400 cells) --
averaged with error bars, since F's 140 cells alone give real sampling noise around the
target (std ≈ 1/sqrt(140) ≈ 0.085 for a true correlation of 0).

In [ ]:
def sweep_correlation(build_fn, targets, n_keys=8, base_seed=0):
    """
    build_fn(key, target) -> (F_sel, J_sel). Returns a DataFrame with one row per target:
    mean/std of achieved Pearson r(F_sel, F_viab) and r(J_sel, J_viab) over n_keys draws.
    """
    rows = []
    for target in targets:
        r_Fs, r_Js = [], []
        for i in range(n_keys):
            key = jax.random.key(base_seed * 1000 + i)
            F_sel, J_sel = build_fn(key, target)
            r_Fs.append(pearson(np.asarray(F_viab).ravel(), np.asarray(F_sel).ravel()))
            r_Js.append(pearson(np.asarray(J_viab)[off_diag].ravel(), np.asarray(J_sel)[off_diag].ravel()))
        rows.append(dict(target=target, r_F_mean=np.mean(r_Fs), r_F_std=np.std(r_Fs),
                          r_J_mean=np.mean(r_Js), r_J_std=np.std(r_Js)))
    return pd.DataFrame(rows)


corr_targets = np.linspace(-1.0, 1.0, 9)
corr_sweep   = sweep_correlation(
    lambda key, t: initialize_correlated_weights(key, F_viab, J_viab, correlation=t),
    corr_targets,
)
corr_sweep

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([-1, 1], [-1, 1], "k--", alpha=0.4, label="target = achieved")
ax.errorbar(corr_sweep["target"], corr_sweep["r_F_mean"], yerr=corr_sweep["r_F_std"],
            fmt="o-", capsize=3, label="F_sel vs F_viab (n=140 cells)")
ax.errorbar(corr_sweep["target"], corr_sweep["r_J_mean"], yerr=corr_sweep["r_J_std"],
            fmt="s-", capsize=3, label="J_sel vs J_viab (n=8,400 cells)")
ax.set_xlabel("target correlation")
ax.set_ylabel("achieved Pearson r (mean ± std over 8 keys)")
ax.set_title("initialize_correlated_weights: target vs achieved correlation")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

### What 3 correlation levels actually look like

In [ ]:
def compare_F(F_viab, F_sel, title):
    """Same 3-panel recipe as AAV9_profile_model.ipynb's F comparisons: two heatmaps
    (shared RdBu_r scale) + a raw-entry scatter with Pearson r."""
    F_viab, F_sel = np.asarray(F_viab), np.asarray(F_sel)
    vmax = max(np.abs(F_viab).max(), np.abs(F_sel).max())

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, F, panel_title in [(axes[0], F_viab, "F_viab"), (axes[1], F_sel, "F_sel")]:
        im = ax.imshow(F, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        ax.set_title(panel_title)
        ax.set_xlabel("Position"); ax.set_ylabel("Amino acid")
        ax.set_xticks(range(NUM_POSITIONS)); ax.set_xticklabels(range(1, NUM_POSITIONS + 1))
        ax.set_yticks(range(NUM_AMINO_ACIDS)); ax.set_yticklabels(AA_LABELS)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    r = pearson(F_viab.ravel(), F_sel.ravel())
    axes[2].scatter(F_viab.ravel(), F_sel.ravel(), s=14, alpha=0.6)
    lims = [min(F_viab.min(), F_sel.min()), max(F_viab.max(), F_sel.max())]
    axes[2].plot(lims, lims, "k--", alpha=0.5)
    axes[2].set_xlabel("F_viab"); axes[2].set_ylabel("F_sel")
    axes[2].set_title(f"Pearson r = {r:.3f}")

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


key_demo = jax.random.key(0)
for target in [0.9, 0.3, -0.9]:
    F_sel, J_sel = initialize_correlated_weights(key_demo, F_viab, J_viab, correlation=target)
    compare_F(F_viab, F_sel, f"initialize_correlated_weights, correlation={target:+.1f}")

## 3. `initialize_anticorrelated_weights`

In [ ]:
anti_targets = np.linspace(0.0, 1.0, 6)
anti_sweep = sweep_correlation(
    lambda key, t: initialize_anticorrelated_weights(key, F_viab, J_viab, anticorrelation=t),
    anti_targets,
)
anti_sweep["expected_r"] = -anti_sweep["target"]
anti_sweep

In [ ]:
F_anti, J_anti = initialize_anticorrelated_weights(key_demo, F_viab, J_viab, anticorrelation=0.8)
compare_F(F_viab, F_anti, "initialize_anticorrelated_weights, anticorrelation=0.8")

## 4. `initialize_indep_weights` — same values, no trend

Instead of drawing fresh Gaussian noise (which would only be an approximately-r=0 case of
the correlated construction), this reshuffles F_viab/J_viab's OWN values -- so F_sel keeps
F_viab's exact real distribution, just moved to different cells.

In [ ]:
F_indep, J_indep = initialize_indep_weights(key_demo, F_viab, J_viab)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(np.asarray(F_viab).ravel(), bins=20, alpha=0.5, label="F_viab")
axes[0].hist(np.asarray(F_indep).ravel(), bins=20, alpha=0.5, label="F_sel (permuted)")
axes[0].set_title("Same values, reshuffled -- identical histogram")
axes[0].set_xlabel("weight value"); axes[0].legend()

r_over_seeds = [
    pearson(np.asarray(F_viab).ravel(),
            np.asarray(initialize_indep_weights(jax.random.key(s), F_viab, J_viab)[0]).ravel())
    for s in range(30)
]
axes[1].hist(r_over_seeds, bins=15, color="tab:green", alpha=0.7)
axes[1].axvline(0, color="k", linestyle="--")
axes[1].set_title(f"Achieved r(F_sel, F_viab) over 30 keys\nmean={np.mean(r_over_seeds):+.3f}, std={np.std(r_over_seeds):.3f}")
axes[1].set_xlabel("Pearson r")
fig.tight_layout()
plt.show()

compare_F(F_viab, F_indep, "initialize_indep_weights (one example draw)")

## 5. Same correlation, different `key`s

`key` is the only source of randomness -- same `key` always gives the exact same F_sel/J_sel
for a given F_viab/J_viab/correlation; different keys give independent draws that still hit
the same TARGET correlation.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, seed in zip(axes, [0, 1, 2, 0]):  # last one repeats seed 0 -> should match the first exactly
    F_sel, _ = initialize_correlated_weights(jax.random.key(seed), F_viab, J_viab, correlation=0.7)
    r = pearson(np.asarray(F_viab).ravel(), np.asarray(F_sel).ravel())
    vmax = max(np.abs(np.asarray(F_viab)).max(), np.abs(np.asarray(F_sel)).max())
    im = ax.imshow(np.asarray(F_sel), aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(f"key={seed}  (r={r:+.3f})")
    ax.set_xlabel("Position"); ax.set_ylabel("Amino acid")
fig.suptitle("initialize_correlated_weights(correlation=0.7) across keys -- same target, different draw\n"
             "(4th panel repeats key=0 -- identical to the 1st, confirming reproducibility)")
fig.tight_layout()
plt.show()

F_a, _ = initialize_correlated_weights(jax.random.key(0), F_viab, J_viab, correlation=0.7)
F_b, _ = initialize_correlated_weights(jax.random.key(0), F_viab, J_viab, correlation=0.7)
print("Same key -> identical F_sel:", bool(np.allclose(np.asarray(F_a), np.asarray(F_b))))

## 6. Payoff — does this actually change simulated directed evolution?

Three `ProtocolV2` instances sharing the exact SAME `F_viab`/`J_viab`/`sequences` -- only
`F_sel`/`J_sel` differ (correlated r=+0.8 / anticorrelated r=-0.8 / indep). If viability and
selectivity favor the SAME amino acids (correlated), a variant that survives production also
tends to survive selectivity -- less overall attrition. If they favor OPPOSITE amino acids
(anticorrelated), variants keep getting filtered out by one stage right after barely passing
the other -- a much harsher combined bottleneck.

In [ ]:
key = jax.random.key(0)
key, k_seq = jax.random.split(key)

num_sequences = 50_000
sequences = jax.random.randint(k_seq, shape=(num_sequences, NUM_POSITIONS), minval=0, maxval=NUM_AMINO_ACIDS)

regimes = {
    "correlated (r=+0.8)":     initialize_correlated_weights(jax.random.key(10), F_viab, J_viab, correlation=0.8),
    "anticorrelated (r=-0.8)": initialize_anticorrelated_weights(jax.random.key(11), F_viab, J_viab, anticorrelation=0.8),
    "indep (permuted)":        initialize_indep_weights(jax.random.key(12), F_viab, J_viab),
}

protocols = {}
for name, (F_sel, J_sel) in regimes.items():
    protocols[name] = ProtocolV2(multinomialNGS=True, 
        N0=100_000_000, N1=N1, dilution_factor=10, sequences=sequences, D=D,
        F_viab=F_viab, J_viab=J_viab, F_sel=F_sel, J_sel=J_sel,
        noise_viab=0.1, noise_sel=0.1,
    )

print("Built 3 protocols, sharing the SAME F_viab/J_viab/sequences -- only F_sel/J_sel differ.")

In [ ]:
summary_rows = []
for name, protocol in protocols.items():
    viab_scores = np.array(protocol.compute_score(protocol.F_viab, protocol.J_viab))
    sel_scores  = np.array(protocol.compute_score(protocol.F_sel,  protocol.J_sel))
    score_corr  = pearson(viab_scores, sel_scores)
    p10         = precision_at_k(viab_scores, sel_scores, k_frac=0.10)

    protocol.loop_DE()
    lambda4 = np.array(protocol.lambda4)

    summary_rows.append(dict(
        regime=name,
        score_corr=score_corr,
        top10pct_overlap=p10,
        surviving=int((lambda4 > 0).sum()),
        pct_surviving=100 * (lambda4 > 0).mean(),
        entropy=shannon_entropy(lambda4),
    ))

summary = pd.DataFrame(summary_rows).set_index("regime")
summary

`score_corr` is the REALIZED correlation between viability and selectivity SCORES (F + J
combined) -- should track the weight-level `correlation`/`anticorrelation` target fairly
closely. `top10pct_overlap` is precision@10%: what fraction of the top-10%-by-viability
sequences are ALSO top-10%-by-selectivity (random baseline = 0.10) -- directly measures how
much passing one filter helps with the other. `surviving`/`entropy` are the actual simulated
outcome after one full directed-evolution round.

Below: `plot_all_lambda_scores` (already in `analysisV1.py`) per regime -- viability vs
selectivity score, colored by abundance at every pipeline stage. Watch how the "surviving"
(bright) region moves: diagonal-shaped survivors under correlation, squeezed into a corner
under anticorrelation.

In [ ]:
for name, protocol in protocols.items():
    _ = plot_all_lambda_scores(protocol, title=f"{name} -- viability vs selectivity score, by pipeline stage")
    plt.show()

## 7. Bonus: does it compound over several rounds?

Same 3 regimes, run through `N_loop_DE` for a few successive directed-evolution rounds --
does the anticorrelated bottleneck keep collapsing diversity round after round, or does it
stabilize?

In [ ]:
N_ROUNDS = 5

trend_rows = []
for name, (F_sel, J_sel) in regimes.items():
    protocol = ProtocolV2(multinomialNGS=True, 
        N0=100_000_000, N1=N1, dilution_factor=10, sequences=sequences, D=D,
        F_viab=F_viab, J_viab=J_viab, F_sel=F_sel, J_sel=J_sel,
        noise_viab=0.1, noise_sel=0.1,
    )
    rounds = protocol.N_loop_DE(N_ROUNDS)
    for round_idx, (bio_row, ngs_row) in enumerate(rounds, start=1):
        lambda0_round = np.array(bio_row[0])
        trend_rows.append(dict(regime=name, round=round_idx,
                                entropy=shannon_entropy(lambda0_round),
                                surviving=int((lambda0_round > 0).sum())))

trend = pd.DataFrame(trend_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name in regimes:
    sub = trend[trend["regime"] == name]
    axes[0].plot(sub["round"], sub["entropy"], marker="o", label=name)
    axes[1].plot(sub["round"], sub["surviving"], marker="o", label=name)
axes[0].set_xlabel("DE round"); axes[0].set_ylabel("Shannon entropy of lambda0"); axes[0].set_title("Diversity (entropy)")
axes[1].set_xlabel("DE round"); axes[1].set_ylabel("# surviving sequences"); axes[1].set_title("Diversity (survivor count)")
for ax in axes:
    ax.legend(); ax.grid(True, linestyle="--", alpha=0.3)
fig.suptitle(f"Library diversity over {N_ROUNDS} directed-evolution rounds, by F_sel/J_sel regime")
fig.tight_layout()
plt.show()